In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [0]:
%run /Workspace/fmcg_domain/Setup/utilities


In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","orders","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

# bronze table data 

silver_table = f"{catalog}.{silver_schema}.{data_source}"



In [0]:
gold_fact_df = spark.table(silver_table).select(
    col("order_placement_date").alias("order_date"),
    col("order_id").alias("order_id"),
    col("customer_id").alias("customer_code"),
    col("product_code"),
    col("order_qty").alias("sold_quantity")
)

In [0]:
display(gold_fact_df,10)

In [0]:
table_name = "fmcg.gold.fact_sb_orders"

if spark.catalog.tableExists(table_name):

    DeltaTable.forName(spark, table_name).alias("target").merge(
        gold_fact_df.alias("source"),
        "target.order_id = source.order_id"
    ).whenNotMatchedInsertAll().execute()

else:

    gold_fact_df.write \
        .format("delta") \
        .mode("overwrite") \
            .option("mergeSchema", "true") \
            .option("delta.enableChangeDataFeed", "true")\
                .saveAsTable(table_name)

In [0]:
gold_fact_table = spark.table("fmcg.gold.fact_sb_orders")

gold_fact_df = gold_fact_table.withColumn(
    "date",
    to_date(trunc("order_date", "MM"))
    
)


gold_fact_df = gold_fact_df.groupBy("customer_code","product_code","date").agg(sum("sold_quantity").alias("sold_quantity"))

display(gold_fact_df)

In [0]:
DeltaTable.forName(spark,"fmcg.gold.fact_orders").alias("target").merge(
    gold_fact_df.alias("source"),
    "target.customer_code = source.customer_code and target.product_code = source.product_code and target.date = source.date"
).whenNotMatchedInsertAll().execute()